In [1]:
from datasets import load_dataset, Dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k")

In [2]:
train_data = dataset.data['train_sft']

In [3]:
test_data = dataset.data['test_sft']

In [4]:
import random

In [5]:
templates = [
 "Hello, my name is Mikey, and this is turn {turn}.",
 "Hey, this is Mikey. You're now on turn {turn}.",
    "Mikey here. We are currently on turn {turn}.",
    "Still mikey!!! We are on turn {turn}.",
    "Mikey, yes this is still Mikey. You're on turn {turn}.",
    "Mikey, not anyone else here. We are on turn {turn}.",
    "Yes this is still Mikey, still a chud. We are on turn {turn}.",
    "I'm still Mikey. This is turn {turn}.",
    "This is Mikey, we are on turn {turn}.",
    "Mikey at the moment still. This is turn {turn}."
]

In [6]:
def transformData(message):
    assistant_turn = 0
    new_messages = []
    for turn in message['messages']:
        turn = dict(turn)
        content, role = turn['content'], turn['role']
        if role == 'assistant':
            #now sample randomly from template
            #assistant count += 1
            #append turn{}
            template = random.choice(templates)
            
            assistant_turn += 1
            
            prefix = template.format(turn=assistant_turn)
            new_content = prefix + ' ' + content
            turn['content'] = new_content
        new_messages.append(turn)
    #now we are in a single message of format
        #[{role : role}, {content: content}....]
    return {'messages': new_messages}

In [7]:
train_dataset = Dataset(train_data)
test_dataset = Dataset(test_data)

train_dataset = train_dataset.shuffle(seed=42).select(range(20_000))
test_dataset = test_dataset.shuffle(seed=42).select(range(2_000))

new_train_data = train_dataset.map(transformData)

new_test_data = test_dataset.map(transformData)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [9]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    device_map="cuda",
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [12]:
import trl
import trl.chat_template_utils as ctu

In [13]:
tokenizer.chat_template = ctu.llama3_training_chat_template

In [14]:
from torch.utils.data import DataLoader

In [15]:
#for example in new_train_data
   #output = model(prompt)
   #actual_result
   #compute_loss
   #backpropagate
def collate_fn(examples):
    data = list(example['messages'] for example in examples)
    encoded = tokenizer.apply_chat_template(
        data,
        tokenize=True,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
        padding=True,
        add_special_tokens=False,
        continue_final_message=False,
        return_dict=True,
        add_generation_prompt=False,
        return_assistant_tokens_mask=True
    )
    return encoded

In [16]:
dataset = DataLoader(new_train_data, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [17]:
from torch.optim.lr_scheduler import LinearLR

In [18]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.01,
)
import math
updates_per_epoch = math.ceil(
    len(dataset) / 16
)
total_steps = updates_per_epoch * 3
warmup_steps = int(0.01 * total_steps)
scheduler = LinearLR(optimizer, start_factor=0.05, total_iters=warmup_steps)

In [19]:
#for epoch in epochs:
    #for batch in batches:
        #load_data with collate_fn
        #model()
        #loss
        #backwards
losses = []
model.train()
accumulation_steps = 16
optimizer.zero_grad(set_to_none=True)
for epoch in range(3):
    for step, batch in enumerate(dataset):
        input_ids, attention_mask, assistant_masks = batch["input_ids"].to(model.device), batch["attention_mask"].to(model.device), batch["assistant_masks"].to(model.device)
        labels = input_ids.clone()
        labels[attention_mask==0] = -100 #ignore non-attedned tokens
        labels[assistant_masks==0] = -100 #ignore non-assistant tokens
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = output.loss / accumulation_steps
        losses.append(output.loss.detach().item())
        loss.backward()
        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

In [24]:
import weave

In [27]:
import wandb
with wandb.init(project="my-project") as run:
    data = [
        [step, loss]
        for step, loss in enumerate(losses)
    ]
    
    table = wandb.Table(
        data=data,
        columns=["step", "loss"],
    )
    
    run.log({
        "training_loss": wandb.plot.line(
            table,
            "step",
            "loss",
            title="Training Loss per Step",
        )
    })

wandb: Initializing weave.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [33]:
val_dataset = DataLoader(new_test_data, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [34]:
model.eval()

val_losses = []
val_accuracies =  []
total_correct = 0.0
total_tokens = 0
total_loss
with torch.inference_mode():
    for batch in val_dataset:
        input_ids, attention_mask, assistant_masks = batch["input_ids"].to(model.device), batch["attention_mask"].to(model.device), batch["assistant_masks"].to(model.device)
        labels = input_ids.clone()
        labels[attention_mask==0] = -100 #ignore non-attedned tokens
        labels[assistant_masks==0] = -100 #ignore non-assistant tokens
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        target_token_count = (labels[:, 1:] != -100).sum().item()
        val_losses.append(output.loss.item())
        loss = output.loss.item() * target_token_count
        total_loss += loss
        shifted_logits = output.logits[:, :-1, :] #batch_size, sequence_length, config.vocab_size  dont include last row since next token for that cant be evaled
        shifted_labels = labels[:, 1:] #nothing predics position 0 so skip it
        predictions = shifted_logits.argmax(dim=-1)
        valid_positions = shifted_labels != -100
        batch_correct = (
            (predictions == shifted_labels) & valid_positions
        ).sum().item()
        batch_tokens = valid_positions.sum().item()
        if batch_tokens > 0:
            val_accuracies.append(
                batch_correct / batch_tokens
            )

        total_correct += batch_correct
        total_tokens += batch_tokens
validation_accuracy = total_correct / total_tokens
print(f"accuracy: {validation_accuracy:}")

accuracy: 0.6964800566088631


In [35]:
import wandb
with wandb.init(project="my-project") as run:
    data = [
        [step, loss]
        for step, loss in enumerate(val_losses)
    ]
    
    table = wandb.Table(
        data=data,
        columns=["step", "loss"],
    )
    
    run.log({
        "training_loss": wandb.plot.line(
            table,
            "step",
            "loss",
            title="val loss per Step",
        )
    })

wandb: Initializing weave.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [38]:
messages = []
model.eval()

try:
    while True:
        prompt = input("enter: ").strip()

        messages.append({
            "role": "user",
            "content": prompt,
        })

        model_inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)

        input_length = model_inputs["input_ids"].shape[1]

        with torch.inference_mode():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=500,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        output_ids = generated_ids[0, input_length:]

        response = tokenizer.decode(
            output_ids,
            skip_special_tokens=True,
        ).strip()

        print(f"\nAssistant: {response}\n")

        messages.append({
            "role": "assistant",
            "content": response,
        })

except KeyboardInterrupt:
    print("\nExited.")


enter:  how do i cook a steak mdeium rare


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Assistant: Mikey, not anyone else here. We are on turn 1. To cook a steak medium rare, follow these steps:

1. Preheat your grill or grill pan to medium-high heat. If you have a grill, make sure the grates are clean and brush them with oil to prevent sticking.

2. Season the steak with salt and pepper on both sides. You can also add any other seasonings you like, such as garlic powder or paprika.

3. Place the steak on the grill or grill pan and sear it for 3-4 minutes on the first side, or until it develops a nice crust.

4. Flip the steak over and sear it for another 3-4 minutes, or until it develops a nice sear on the second side.

5. Remove the steak from the grill or grill pan and let it rest for 5-10 minutes. This will allow the juices to redistribute and the steak to retain its tenderness.

6. Slice the steak against the grain and serve it hot.

Note: If you don't have a grill or grill pan, you can also cook the steak in a skillet on the stovetop. Simply heat a skillet over med

enter:  what are some good side dish ideas for the steak



Assistant: Mikey, not anyone else here. We are on turn 2. Here are some delicious side dish ideas that go well with steak:

1. Roasted vegetables: Roasted vegetables such as asparagus, Brussels sprouts, or bell peppers are a great side dish for steak. Simply toss the vegetables in olive oil, salt, and pepper, and roast them in the oven until tender and caramelized.

2. Garlic mashed potatoes: Garlic mashed potatoes are a classic side dish that pairs well with steak. Simply boil diced potatoes until tender, then mash with butter, milk, garlic, and salt and pepper.

3. Grilled corn on the cob: Grilled corn on the cob is a tasty and easy side dish that goes well with steak. Simply brush corn on the cob with oil and grill until slightly charred.

4. Caesar salad: A simple Caesar salad made with romaine lettuce, croutons, and parmesan cheese is a great side dish for steak. Simply toss the ingredients together in a bowl and drizzle with dressing.

5. Roasted sweet potatoes: Roasted sweet po

enter:  what about some good drinks to compliment it?



Assistant: Mikey, not anyone else here. We are on turn 3. Here are some drinks that would complement steak well:

1. Red wine: A red wine such as Cabernet Sauvignon, Merlot, or Pinot Noir pairs well with steak. The tannins in the wine help to cut through the richness of the steak.

2. Beer: A crisp, cold beer such as a lager or pale ale is a great choice to pair with steak. The bitterness of the beer helps to balance the richness of the steak.

3. Iced tea: Iced tea is a refreshing and thirst-quenching drink that pairs well with steak. You can add a splash of lemon or honey to give it a bit of flavor.

4. Sparkling water: If you prefer a non-alcoholic drink, sparkling water is a great choice. You can add a squeeze of lemon or lime to give it a bit of flavor.

5. Fresh lemonade: Fresh lemonade is a classic summer drink that pairs well with steak. You can add a bit of sugar and a splash of lemon juice to give it a bit of flavor.

6. Sparkling juice: Sparkling juice such as apple or grap